# ST446 WT2026 - Assigment 1


**Task information:** In this assignment, you will analyse global COVID-19 pandemic data. You will progressively move from low-level distributed processing (Spark RDDs) to structured analytics (Spark DataFrames and Spark SQL) and finally to NoSQL analytics using Bigtable.

The assignment consists of **4 parts**, as follows:
* **Part A:** Spark RDD (4 questions)
* **Part B:** Spark DataFrames (3 questions)
* **Part C:** Spark SQL (2 questions)
* **Part D:** Bigtable (2 questions)

Your **overall goal** is not only to compute results, but to justify analytical choices, understand trade-offs between APIs, and reflect on how AI tools (if used) supported or influenced your reasoning.

<hr>

## Setup

In [1]:
# Python libraries
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime

# Spark libraries
from pyspark.sql import SparkSession
# Spark SQL data types
from pyspark.sql.types import *
# Spark SQL functions
import pyspark.sql.functions as sql_f

<hr>

## Data

Check the data source/provider and information about the dataset, especially metadata.

* Dataset: **COVID-19 Pandemic**
* Data source: Our World in Data
* Latest version (including metadata/column information): https://docs.owid.io/projects/covid/en/latest/dataset.html
* Project information: https://ourworldindata.org/coronavirus


In [2]:
# -----------------------------------------------
# Dataset download and Hadoop ingestion
# -----------------------------------------------
!wget https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv
!hadoop fs -put -f owid-covid-data.csv /

--2026-03-03 17:51:41--  https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 98391483 (94M) [text/plain]
Saving to: ‘owid-covid-data.csv’

owid-covid-data.csv 100%[===================>]  93.83M   503MB/s    in 0.2s    

2026-03-03 17:51:42 (503 MB/s) - ‘owid-covid-data.csv’ saved [98391483/98391483]



In [3]:
# -----------------------------------------------
# Check Hadoop masternode
# -----------------------------------------------
!hadoop fs -ls /

Found 4 items
-rw-r--r--   2 root hadoop   98391483 2026-03-03 17:51 /owid-covid-data.csv
drwxrwxrwt   - hdfs hadoop          0 2026-03-03 16:47 /tmp
drwxrwxrwt   - hdfs hadoop          0 2026-03-03 16:48 /user
drwxrwxrwt   - hdfs hadoop          0 2026-03-03 16:47 /var


In [4]:
# -----------------------------------------------
# HDFS file path
# Adjust to match your cluster name and file path
# -----------------------------------------------
filename = "hdfs://st446-a1-m:8020/owid-covid-data.csv"

<hr>

## Part A — Spark RDDs (4 questions)

**Goal:** Understand distributed data processing, key/value transformations, aggregation, and ranking logic.

---

### Dataset loading & exploration

As usual in most analytical scenarios, you can **load the raw data into a RDD** and keep it unchanged throughout your analytical pipeline. For each specific task, this RDD can be loaded into a new RDD and subject to the required transformations/actions.

In [5]:
# Raw RDD
raw_rdd = spark.sparkContext.textFile(filename)
# Check header (column names)
header = raw_rdd.first()
columns = header.split(",")
# Print out column names
for i, col_name in enumerate(columns):
    print(f"Index: {i}: {col_name}")

Index: 0: iso_code
Index: 1: continent
Index: 2: location
Index: 3: date
Index: 4: total_cases
Index: 5: new_cases
Index: 6: new_cases_smoothed
Index: 7: total_deaths
Index: 8: new_deaths
Index: 9: new_deaths_smoothed
Index: 10: total_cases_per_million
Index: 11: new_cases_per_million
Index: 12: new_cases_smoothed_per_million
Index: 13: total_deaths_per_million
Index: 14: new_deaths_per_million
Index: 15: new_deaths_smoothed_per_million
Index: 16: reproduction_rate
Index: 17: icu_patients
Index: 18: icu_patients_per_million
Index: 19: hosp_patients
Index: 20: hosp_patients_per_million
Index: 21: weekly_icu_admissions
Index: 22: weekly_icu_admissions_per_million
Index: 23: weekly_hosp_admissions
Index: 24: weekly_hosp_admissions_per_million
Index: 25: total_tests
Index: 26: new_tests
Index: 27: total_tests_per_thousand
Index: 28: new_tests_per_thousand
Index: 29: new_tests_smoothed
Index: 30: new_tests_smoothed_per_thousand
Index: 31: positive_rate
Index: 32: tests_per_case
Index: 33: t

---

### Question 1 - Baseline RDD

Using the **raw RDD** loaded in the previous cell:
* create a reduced **baseline RDD** named ` covid_rdd` to answer all questions in Part A
* show three samples from the baseline RDD

The baseline RDD needs only a few columns:

```
# | Index    | Column             |
# | -------- | ------------------ |
# | 0        | iso_code           |
# | 1        | continent          |
# | 2        | location           |
# | 3        | date               |
# | 4        | total_cases        |
# | 5        | new_cases          |
# | 17       | icu_patients       |
# | 19       | hosp_patients      |
# | 34       | total_vaccinations |
# | 38       | new_vaccinations   |
# | 62       | population         |

```

**IMPORTANT:** OWID frequently adds new columns to the dataset. **You must verify column positions using the dataset metadata before hard-coding indices. Your code should not assume indices blindly without validating the schema**.

**OWID Dataset — baseline RDD assumptions:**
* `iso_code` refers to three-letter country codes (e.g. USA, FRA, DEU) or to prefix `OWID_` for non-country level aggregated data (e.g. continents like 'Europe' etc)
* Rows where `continent` is empty (or `None`) can also correspond to non-country aggregates (e.g. European Union, Internatioanl, income groups)
* `total_cases` and `total_vaccinations` are cumulative, so `max(total_cases)` gives final count
* Dates are in `YYYY-MM-DD` format (lexicographically sortable)

**Requirements and Constraints:**
* Verify column positions in the raw RDD before hard-coding indices
* Baseline RDD must be named `covid_rdd`
* Remove headers when generating the baseline RDD
* Map only the necessary columns from the raw RDD into `covid_rdd`

**Expected output:** the baseline RDD should have the following structure:

```
# | Index    | Column             |
# | -------- | ------------------ |
# | 0        | iso_code           |
# | 1        | continent          |
# | 2        | location           |
# | 3        | date               |
# | 4        | total_cases        |
# | 5        | new_cases          |
# | 6        | icu_patients       |
# | 7        | hosp_patients      |
# | 8        | total_vaccinations |
# | 9        | new_vaccinations   |
# | 10       | population         |
```

A sample should look like:
```
[('AFG',
  'Asia',
  'Afghanistan',
  '2020-01-05',
  '0',
  '0',
  '',
  '',
  '',
  '',
  '41128772')
```

In [6]:
## CODE

# Extracting header and columns
header = raw_rdd.first()
columns = header.split(",")

# Finding and verifying indices of specified columns in raw rdd
needed_cols = [
    "iso_code",
    "continent",
    "location",
    "date",
    "total_cases",
    "new_cases",
    "icu_patients",
    "hosp_patients",
    "total_vaccinations",
    "new_vaccinations",
    "population"
]

for col in needed_cols:
    print(f"{col}: {columns.index(col)}")

# Creation of baseline rdd without header and mapped columns
covid_rdd = raw_rdd \
    .filter(lambda row: row != header) \
    .map(lambda row: row.split(",")) \
    .map(lambda r: (
        r[0],   # iso_code
        r[1],   # continent
        r[2],   # location
        r[3],   # date
        r[4],   # total_cases
        r[5],   # new_cases
        r[17],  # icu_patients
        r[19],  # hosp_patients
        r[34],  # total_vaccinations
        r[38],  # new_vaccinations
        r[62]   # population
    ))

# Printing a sample from baseline rdd
covid_rdd.take(1)

iso_code: 0
continent: 1
location: 2
date: 3
total_cases: 4
new_cases: 5
icu_patients: 17
hosp_patients: 19
total_vaccinations: 34
new_vaccinations: 38
population: 62


[('AFG',
  'Asia',
  'Afghanistan',
  '2020-01-05',
  '0',
  '0',
  '',
  '',
  '',
  '',
  '41128772')]

---

### Question 2: Hierarchical Aggregation (Continent -> Country)

Using the **baseline RDD**:
* Compute the total number of confirmed COVID-19 cases per continent level
* For each continent, identify the top 2 countries with the highest cumulative number of confirmed cases
* Show results from both RDDs

**Requirements and Constraints:**
* Use RDD transformations and actions only (no DataFrames or SQL)
* Filter out rows that correspond to aggregates and rows with missing continent
* Use the maximum cumulative total cases per country before aggregating at continent level

**Expected output:**
* One RDD with total cases per continent: `[(continent, total cases), ...]`
* One RDD with top 2 countries per continent by total cases, descending order: `[(continent, [(country1, total cases), (country2, total cases)]), ...]`
* Show results from both RDDs



In [7]:
# CODE

# covid_rdd schema
ISO, CONT, LOC, DATE, TOTAL_CASES = 0, 1, 2, 3, 4

# country-level only & no aggregate entries included
country = covid_rdd.filter(lambda r: r[CONT] and not r[ISO].startswith("OWID"))

# Max cumulative cases per (continent, country) where 
    # key: (continent, country)  value: max(total_cases)
max_by_country = (
    country
    .map(lambda r: ((r[CONT], r[LOC]), float(r[TOTAL_CASES]) if r[TOTAL_CASES] else 0.0))
    .reduceByKey(max)
)

# Total cases per continent = sum of country maximum totals
continent_totals = (
    max_by_country
    .map(lambda k_v: (k_v[0][0], k_v[1]))   # (continent, max_cases)
    .reduceByKey(lambda x, y: x + y)
)

# Top 2 countries per continent
top2 = (
    max_by_country
    .map(lambda k_v: (k_v[0][0], (k_v[0][1], k_v[1])))   # (continent, (country, cases))
    .groupByKey()
    .mapValues(lambda vals: sorted(vals, key=lambda t: -t[1])[:2])
)

# Printing results

print("Continent totals:")
print(continent_totals.sortBy(lambda x: -x[1]).collect())

print("\nTop 2 countries per continent:")
print(top2.sortByKey().collect())

Continent totals:


[('Asia', 301532347.0), ('Europe', 252642589.0), ('North America', 124492666.0), ('South America', 68809418.0), ('Oceania', 15003352.0), ('Africa', 13145540.0)]

Top 2 countries per continent:
[('Africa', [('South Africa', 4072765.0), ('Morocco', 1279115.0)]), ('Asia', [('China', 99373219.0), ('India', 45041748.0)]), ('Europe', [('France', 38997490.0), ('Germany', 38437756.0)]), ('North America', [('United States', 103436829.0), ('Mexico', 7619458.0)]), ('Oceania', [('Australia', 11861161.0), ('New Zealand', 2639048.0)]), ('South America', [('Brazil', 37511921.0), ('Argentina', 10101218.0)])]


<hr>

### Question 3 – Temporal Growth Analysis

Using the **baseline RDD**:
* Restrict the dataset to the period March–June 2021
* Compute weekly total new COVID-19 cases per country
* For each continent, identify the single country with the highest week-over-week growth in new cases during this period
* Show the resulting RDD

**Requirements and Constraints:**
* Use RDD transformations and actions only (no DataFrames or SQL)
* Filter out missing or null values from the required columns
* Define week using ISO week number derived from the date. Week-over-week growth is defined as: `growth = (current_week_total − previous_week_total)`
* Growth should be computed within each country only

**Expected output:** RDD with `[(continent, (country, highest week-over-week growth)), ...]`

In [8]:
# CODE

ISO, CONT, LOC, DATE, TOTAL_CASES, NEW_CASES = 0, 1, 2, 3, 4, 5


# Filtering March-June 2021 on country-level with new cases
period = covid_rdd.filter(lambda r: r[CONT] and not r[ISO].startswith("OWID")) \
                  .filter(lambda r: "2021-03-01" <= r[DATE] <= "2021-06-30") \
                  .filter(lambda r: r[NEW_CASES] not in (None, ""))


# Weekly total new cases per country
    # Key: (continent, country, iso_year, iso_week)  Value: weekly sum(new_cases)   
weekly_country = period.map(lambda r: (
                        (r[CONT],
                         r[LOC],
                         datetime.strptime(r[DATE], "%Y-%m-%d").isocalendar()[0],
                         datetime.strptime(r[DATE], "%Y-%m-%d").isocalendar()[1]),
                        float(r[NEW_CASES])
                    )) \
                    .reduceByKey(lambda x, y: x + y)


# Computing max week-over-week growth within each country

# First grouped weeks by country (continent,country) and sort them
country_weeks = (weekly_country
    .map(lambda k_v: ((k_v[0][0], k_v[0][1]), (k_v[0][2], k_v[0][3], k_v[1])))
    .groupByKey()
    .mapValues(lambda rows: sorted(rows, key=lambda x: (x[0], x[1])))
)

# Max week-over-week growth within each country
# key:((continent, country), value: max_growth)
max_growth_by_country = (country_weeks
    .filter(lambda k_v: len(k_v[1]) >= 2)
    .mapValues(lambda s: max(s[i][2] - s[i-1][2] for i in range(1, len(s))))
)


# Selecting single country with highest growth per continent
result_rdd = (max_growth_by_country
    .map(lambda k_v: (k_v[0][0], (k_v[0][1], k_v[1])))
    .reduceByKey(lambda x, y: x if x[1] >= y[1] else y)
)

# Displaying results
result_rdd.collect()


[('Africa', ('South Africa', 32958.0)),
 ('South America', ('Brazil', 80556.0)),
 ('Asia', ('India', 742759.0)),
 ('Europe', ('France', 40845.0)),
 ('Oceania', ('Papua New Guinea', 782.0)),
 ('North America', ('United States', 38267.0))]

<hr>

### Question 4: Vaccination Intensity Ranking

Using the **baseline RDD**:
* Compute an RDD with the average daily number of new COVID-19 vaccinations per country.
* Return the top 10 countries by average daily vaccinations.

**Requirements & Constraints:**
* Use RDD transformations and actions only
* Use valid (not null/missing) daily new vaccinations
* Do not use cumulative vaccination columns
* Include only country-level records (exclude continents, world totals, and income groups)
* Include only countries with at least 50 days of valid reported vaccination data
* Round numeric outputs to two decimal places

**Expected output:** RDD with `[(country, average daily vaccinations), ...]`

In [9]:
# CODE

ISO, CONT, LOC, NEW_VAX = 0, 1, 2, 9

# Keeping only country-level rows and valid new_vaccinations
vax_rows = covid_rdd.filter(lambda r: r[CONT] and not r[ISO].startswith("OWID")) \
                    .filter(lambda r: r[NEW_VAX] not in (None, ""))

# Map to (country, (sum_new_vax, day_count))
country_sum_count = vax_rows.map(lambda r: (r[LOC], (float(r[NEW_VAX]), 1))) \
                            .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))

# Keeping only countries with at least 50 valid reporting days
country_sum_count_50 = country_sum_count.filter(lambda k_v: k_v[1][1] >= 50)

# Computing average daily new vaccinations rounded to 2 decimals
country_avg = country_sum_count_50.map(lambda k_v: (k_v[0], round(k_v[1][0] / k_v[1][1], 2)))

# Displaying Top 10 by average daily vaccinations (descending)
top10_avg_daily_vax = country_avg.takeOrdered(10, key=lambda k_v: -k_v[1])

top10_avg_daily_vax


[('China', 5147424.47),
 ('India', 1731265.63),
 ('Indonesia', 808177.92),
 ('United States', 771588.55),
 ('Brazil', 724200.64),
 ('Pakistan', 587713.4),
 ('Japan', 489532.5),
 ('Mexico', 444550.72),
 ('Bangladesh', 441370.03),
 ('Philippines', 426459.62)]

<hr>

## Part B - Spark DataFrames (2 questions)

**Goal:** Understand structured analytics using transformations, aggregation, and ranking logic.

---

### Question 5: Data loading & baseline DataFrame

When loading data into Spark DataFrames, we usually define a mapping schema, based on `StructType` and `StructField`. This requires us to know the structure of the dataset and ensure that `StructField` names match the column names in the dataset; otherwise, NULL values are loaded. There are also other potential problems causing **column misalignment**, such as wrong separators and date formats.

For this part of the assignment, you must **create a baseline DataFrame** named `covid_df`:
* load the original dataset into a raw DataFrame (e.g., `raw_df`)
* filter out only the relevant columns (see table below) into a baseline DataFrame (e.g., `covid_df`)
* show three samples

**Requirements and Constraints:**
* instead of defining a schema, ask Spark to infer the schema for you. This is safer and avoids any potential column misalignment arising from the data
* note that we need **two additional columns** compared to the baseline RDD used in Part A.
* before attempting any DataFrames questions, print out some sample rows and check the inferred schema

**Expected output:** the baseline DataFrame should have this structure:

```
| Column name          | Description             |
| -------------------- | ----------------------- |
| `iso_code`           | Country codes           |
| `continent`          | Continent name          |
| `location`           | Country name            |
| `date`               | Observation date        |
| `total_cases`        | Cumulative COVID cases  |
| `new_cases`          | New cases               |
| `icu_patients`       | ICU patients            |
| `hosp_patients`      | Hospitalized patients   |
| `total_vaccinations` | Cumulative vaccinations |
| `new_vaccinations`   | Daily vaccinations      |
| `population`         | Total population        |
| `total_deaths`       | Total deaths            |
| `new_deaths`         | New deaths              |
| -------------------- | ----------------------- |
```

A sample will look like:

```
----------+----------+------------+----------+
|iso_code|continent|location |date      |total_cases|new_cases|icu_patients|hosp_patients|total_vaccinations|new_vaccinations|population|total_deaths|new_deaths|
+--------+---------+---------+----------+-----------+---------+------------+-------------+------------------+----------------+----------+------------+----------+
|AUS     |Oceania  |Australia|2021-03-22|29196      |0        |1           |66           |312502            |30542           |26177410  |925         |0         |
```


In [29]:
# CODE

# Reading data into dataframe, inferring schema
raw_df = spark.read.csv(
    "/owid-covid-data.csv",
    header=True,
    inferSchema=True
)

# Checking inferred schema
raw_df.printSchema()
raw_df.show(3, truncate=True)

# Required columns
covid_df = raw_df.select(
    "iso_code",
    "continent",
    "location",
    "date",
    "total_cases",
    "new_cases",
    "icu_patients",
    "hosp_patients",
    "total_vaccinations",
    "new_vaccinations",
    "population",
    "total_deaths",
    "new_deaths"
)

# Printing sample rows
covid_df.show(3, truncate=True)

# Printing expected output row for check
covid_df.filter(
    (covid_df.iso_code == "AUS") & 
    (covid_df.date == "2021-03-22")
).show(truncate=False)


root
 |-- iso_code: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- location: string (nullable = true)
 |-- date: date (nullable = true)
 |-- total_cases: integer (nullable = true)
 |-- new_cases: integer (nullable = true)
 |-- new_cases_smoothed: double (nullable = true)
 |-- total_deaths: integer (nullable = true)
 |-- new_deaths: integer (nullable = true)
 |-- new_deaths_smoothed: double (nullable = true)
 |-- total_cases_per_million: double (nullable = true)
 |-- new_cases_per_million: double (nullable = true)
 |-- new_cases_smoothed_per_million: double (nullable = true)
 |-- total_deaths_per_million: double (nullable = true)
 |-- new_deaths_per_million: double (nullable = true)
 |-- new_deaths_smoothed_per_million: double (nullable = true)
 |-- reproduction_rate: double (nullable = true)
 |-- icu_patients: integer (nullable = true)
 |-- icu_patients_per_million: double (nullable = true)
 |-- hosp_patients: integer (nullable = true)
 |-- hosp_patients_per_mil

+--------+---------+-----------+----------+-----------+---------+------------+-------------+------------------+----------------+----------+------------+----------+
|iso_code|continent|   location|      date|total_cases|new_cases|icu_patients|hosp_patients|total_vaccinations|new_vaccinations|population|total_deaths|new_deaths|
+--------+---------+-----------+----------+-----------+---------+------------+-------------+------------------+----------------+----------+------------+----------+
|     AFG|     Asia|Afghanistan|2020-01-05|          0|        0|        NULL|         NULL|              NULL|            NULL|  41128772|           0|         0|
|     AFG|     Asia|Afghanistan|2020-01-06|          0|        0|        NULL|         NULL|              NULL|            NULL|  41128772|           0|         0|
|     AFG|     Asia|Afghanistan|2020-01-07|          0|        0|        NULL|         NULL|              NULL|            NULL|  41128772|           0|         0|
+--------+------

---

### Question 6: System Pressure Indicator (ICU aggregations)

Using the **baseline DataFrame**:
* Compute a DataFrame with the average ICU patient load per country
* Show the top 10 countries

**Requirements and Constraints:**
* Use only country-level rows (exclude all continent / world aggregates)
* Include only countries with at least 50 non-null ICU observations
* Order results in descending order

**Expected output:** DataFrame with
* `location`: country name
* `icu_obs_count`: count of valid ICU observations
* `avg_icu_patients`: computed average ICU patient load

In [11]:
# CODE

icu_summary = (
    covid_df
    
    # Filtering for only country level observations
    .filter(sql_f.col("continent").isNotNull())
    .filter(~sql_f.col("iso_code").startswith("OWID"))

    # Grouping data at country level
    .groupBy("location")

    # Computing icu_obs_count and avg_icu_patients
    .agg(
        sql_f.count("icu_patients").alias("icu_obs_count"),
        sql_f.avg("icu_patients").alias("avg_icu_patients")
    )

    # Filtering out countries with less than 50 ICU observations
    .filter(sql_f.col("icu_obs_count") >= 50)

    # Ranking by avg ICU load, descending
    .orderBy(sql_f.col("avg_icu_patients").desc())
)

# Printing results
icu_summary.show(10, truncate=False)

+--------------+-------------+------------------+
|location      |icu_obs_count|avg_icu_patients  |
+--------------+-------------+------------------+
|United States |1381         |7703.354815351195 |
|Argentina     |647          |2934.584234930448 |
|France        |1109         |2046.5193868349866|
|Germany       |1194         |1771.2361809045226|
|Spain         |1045         |1060.6            |
|United Kingdom|781          |900.5006402048656 |
|South Africa  |873          |803.6540664375716 |
|Chile         |1248         |795.6642628205128 |
|Japan         |203          |706.8374384236453 |
|Italy         |1627         |695.0503995082975 |
+--------------+-------------+------------------+
only showing top 10 rows



---

### Question 7: Vaccination Momentum vs Population Size

Using the **baseline DataFrame**, for each country, compute:
* total number of vaccinations administered
* average daily vaccinations
* number of days with reported vaccination data
* `vaccinations_per_1000`, given by `(total vaccinations/population) * 1000`

**Requirements:**
* Restrict the dataset to country-level observations only (exclude continents and other aggregates)
* Consider only rows with valid `new_vaccinations` and `population` data
* Include only countries with at least 50 reporting days
* Round numeric outputs to two decimal places
* Order the results by`vaccinations_per_1000` in descending order

**Expected output:** DataFrame with:
* `location`
* `population`
* `total_vaccination`
* `avg_daily_vaccinations`
* `reporting_days`
* `vaccinations_per_1000`



In [12]:
# CODE

# Filtering valid country-level rows
country_vax = covid_df \
    .filter(sql_f.col("continent").isNotNull()) \
    .filter(~sql_f.col("iso_code").startswith("OWID")) \
    .filter(sql_f.col("new_vaccinations").isNotNull()) \
    .filter(sql_f.col("population").isNotNull())

# Aggregating measures per country
vax_summary = country_vax \
    .groupBy("location", "population") \
    .agg(
        sql_f.sum("new_vaccinations").alias("total_vaccination"),
        sql_f.avg("new_vaccinations").alias("avg_daily_vaccinations"),
        sql_f.count("new_vaccinations").alias("reporting_days")
    )

# Keeping the countries with 50 or more reporting days
vax_summary_50 = vax_summary.filter(sql_f.col("reporting_days") >= 50)

# Computing vaccinations per 1,000 people
vax_summary_final = vax_summary_50.withColumn(
    "vaccinations_per_1000",
    (sql_f.col("total_vaccination") / sql_f.col("population")) * 1000
)

# Rounding output
vax_summary_final = vax_summary_final \
    .withColumn("total_vaccination", sql_f.round(sql_f.col("total_vaccination"), 2)) \
    .withColumn("avg_daily_vaccinations", sql_f.round(sql_f.col("avg_daily_vaccinations"), 2)) \
    .withColumn("vaccinations_per_1000", sql_f.round(sql_f.col("vaccinations_per_1000"), 2))

# Ordering by vaccinations_per_1000 descending
vax_summary_final \
    .orderBy(sql_f.col("vaccinations_per_1000").desc()) \
    .show(truncate=False)

+-----------+----------+-----------------+----------------------+--------------+---------------------+
|location   |population|total_vaccination|avg_daily_vaccinations|reporting_days|vaccinations_per_1000|
+-----------+----------+-----------------+----------------------+--------------+---------------------+
|Cuba       |11212198  |38404382         |59634.13              |644           |3425.23              |
|Chile      |19603736  |62671925         |83562.57              |750           |3196.94              |
|Japan      |123951696 |383303948        |489532.5              |783           |3092.37              |
|Gibraltar  |32677     |92234            |501.27                |184           |2822.6               |
|Hong Kong  |7488863   |20839671         |23336.7               |893           |2782.76              |
|Belgium    |11655923  |31458352         |29074.26              |1082          |2698.92              |
|Peru       |34049588  |91408722         |88403.02              |1034    

---

### Question 8: Pandemic Severity Summary

Using the **baseline DataFrame** as a Spark SQL table, create a **country-level summary table** (`severity_df`) that supports cross-country risk comparison. For each country, compute:
* the maximum cumulative number of confirmed COVID-19 cases
* the maximum cumulative number of confirmed COVID-19 deaths
* `case_fatality_ratio` (CFR), defined as `(maximum total deaths / maximum total cases) * 100`

**Requirements:**
* Restrict the analysis to country-level observations only
* Include only rows with valid `total_cases` and `total_deaths` data
* CFR is approximated using maximum cumulative values; this does not account for temporal lag between cases and deaths
* Round numeric outputs to two decimal places
* Order the table by `case_fatality_ratio` in descending order

**Expected output:** `severity_df` DataFrame with:
* `location`
* `max_total_cases`
* `max_total_deaths`
* `case_fatality_ratio`


In [13]:
# CODE

# Temporary View Created
covid_df.createOrReplaceTempView("covid")

# severity_df creation
severity_df = spark.sql("""
SELECT
  location,
  MAX(total_cases)  AS max_total_cases,
  MAX(total_deaths) AS max_total_deaths,
  ROUND((MAX(total_deaths) / MAX(total_cases)) * 100, 2) AS case_fatality_ratio
FROM covid
WHERE
  continent IS NOT NULL
  AND iso_code NOT LIKE 'OWID%'
  AND total_cases IS NOT NULL
  AND total_deaths IS NOT NULL
  AND total_cases > 0
GROUP BY location
ORDER BY case_fatality_ratio DESC
""")

# Displaying df
severity_df.show(truncate=False)

+----------------------+---------------+----------------+-------------------+
|location              |max_total_cases|max_total_deaths|case_fatality_ratio|
+----------------------+---------------+----------------+-------------------+
|Yemen                 |11945          |2159            |18.07              |
|Sudan                 |63993          |5046            |7.89               |
|Syria                 |57423          |3163            |5.51               |
|Somalia               |27334          |1361            |4.98               |
|Peru                  |4526977        |220975          |4.88               |
|Egypt                 |516023         |24830           |4.81               |
|Mexico                |7619458        |334551          |4.39               |
|Bosnia and Herzegovina|403666         |16392           |4.06               |
|Liberia               |8090           |294             |3.63               |
|Afghanistan           |235214         |7998            |3.4    

---

### Question 9: Health System & Vaccination Readiness

Using the **baseline DataFrame** as a Spark SQL table, create a **country-level summary table** (`health_vax_df`) combining indicators of healthcare system burden and vaccination progress. Such summary table would enable comparison of countries' readiness to manage COVID-19.

For each country, compute:
* the average number of hospitalised COVID-19 patients over time
* the average number of ICU COVID-19 patients over time
* vaccinations per 1,000 people (`vax_per_1k`), defined as `(maximum total vaccinations / population) * 1000`

**Requirements and Constraints:**
* Restrict the analysis to country-level observations only
* For each metric, use non-null observations; however, only include countries where all required columns are non-null in the final result
* Round all numeric outputs to two decimal places
* Order the results by `vax_per_1k` in descending order
* Remember that `total_vaccinations` is a cumulative variable

**Expected output:** `health_vax_df` DataFrame with:
* `location`
* `avg_hosp_patients`
* `avg_icu_patients`
* `vax_per_1k`


In [14]:
# CODE

# health_vax dataframe creation
health_vax_df = spark.sql("""
SELECT
    location,
    ROUND(AVG(hosp_patients), 2) AS avg_hosp_patients,
    ROUND(AVG(icu_patients), 2) AS avg_icu_patients,
    ROUND((MAX(total_vaccinations) / population) * 1000, 2) AS vax_per_1k
FROM covid
WHERE
    continent IS NOT NULL
    AND iso_code NOT LIKE 'OWID%'
    AND hosp_patients IS NOT NULL
    AND icu_patients IS NOT NULL
    AND total_vaccinations IS NOT NULL
    AND population IS NOT NULL
GROUP BY location, population
ORDER BY vax_per_1k DESC
""")

health_vax_df.show(truncate=False)

+--------------+-----------------+----------------+----------+
|location      |avg_hosp_patients|avg_icu_patients|vax_per_1k|
+--------------+-----------------+----------------+----------+
|Japan         |14252.59         |986.88          |3095.85   |
|Canada        |4027.83          |336.3           |2663.79   |
|Australia     |1708.11          |121.09          |2647.56   |
|Denmark       |410.33           |27.99           |2542.33   |
|Belgium       |1576.85          |239.44          |2534.81   |
|South Korea   |3764.23          |436.24          |2501.9    |
|Sweden        |848.61           |81.44           |2458.38   |
|Italy         |8512.15          |686.39          |2449.44   |
|Finland       |560.93           |31.7            |2359.41   |
|Portugal      |1341.62          |202.44          |2316.52   |
|Austria       |1219.69          |165.8           |2289.67   |
|France        |18085.09         |1986.96         |2276.98   |
|Netherlands   |607.89           |160.12          |2263

<hr>

## PART D — Google Bigtable

**Goal:** Model a NoSQL (key/value pairs, in this case) table and run analytical queries.

Using the summary tables (`severity_df` and `health_vax_df`) that you computed in Part C, you will:
* Create a Bigtable instance and table.
* Define appropriate column families.
* Load data from summary tables into Bigtable.
* Run analytical queries on Bigtable rows.

---

### Bigtable setup and authentication

[This is based on Week02's seminar example]


In [15]:
# -----------------------------------------------
# Ensure latest version of Bigtable GCP connector
# -----------------------------------------------
!pip install --upgrade google-cloud-bigtable

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.3/540.3 kB 36.8 MB/s eta 0:00:00
  Attempting uninstall: google-cloud-bigtable
    Found existing installation: google-cloud-bigtable 2.21.0
    Uninstalling google-cloud-bigtable-2.21.0:
      Successfully uninstalled google-cloud-bigtable-2.21.0


In [16]:
# -----------------------------------------------
# Client library for authentication in GCP
# -----------------------------------------------
from oauth2client.client import GoogleCredentials

In [17]:
# --------------------------------------------------------
# If Bigtable API is enabled, import necessary libraries
# --------------------------------------------------------
from google.cloud import bigtable
from google.cloud.bigtable import enums
from google.cloud.bigtable.column_family import MaxVersionsGCRule
import datetime

In [18]:
# -----------------------------------------------
# Retrieve GCP credentials
# -----------------------------------------------
credentials = GoogleCredentials.get_application_default()
print(credentials)

In [19]:
# -----------------------------------------------
# Bigtable client connection
# -----------------------------------------------
# REPLACE WITH YOUR GCP PROJECT ID AND DESIRED ZONE
PROJECT_ID = "ecstatic-spirit-485110-q7"       # Replace with your GCP project ID
ZONE = "europe-west2-c"           # Replace with your preferred zone (ideally, the same of Dataproc cluster)

# Instantiate a BigTable client
# admin=True is necessary for writing operations
client = bigtable.Client(project=PROJECT_ID, admin=True)

# Check connection by listing any instances running on Bigtable
instances = client.list_instances()
# Print out all instances; empty if no instances exist
for i in instances[0]:
    print (i.name)

---

### Question 10: Designing and Populating a Bigtable Serving Layer

Using the two summary tables (`severity_df` and `health_vax_df`), design and implement a **Bigtable schema to store country-level COVID-19 performance metrics**.

You must:
* Create a Bigtable instance (`covid-instance`)
* Create a table (`covid_summary`) with appropriate column families (see below)
* Insert data from both Spark SQL summary tables
* Verify that the data was correctly inserted by reading a few sample rows from your Bigtable table

You must design a **denormalised Bigtable schema** with:
* Each row representing one country, identified by the country name (`location`) as the row key
* The following column families:
    * `meta`, with structural country information (`continent` and `population`)
    * `severity`, with pandemic outcome metrics (`max_total_cases`, `max_total_deaths`, `case_fatality_ratio`)
    * `health_vax`, with health system and vaccination metrics (`avg_hosp_patients`, `avg_icu_patients`, `vax_per_1k`)
    
**Requirements & Constraints:**
* Do not create multiple tables (only one denormalized table)
* Do not store all columns in one family
* Do not use joins inside Bigtable (1). All joins must be performed in Spark before writing to Bigtable
* Remember that Bigtable stores numeric values as encoded bytes (`utf-8`)
* Each country must appear only once
* All data must only include country-level rows with complete data entries (no NULLs), consistent with previous tasks

(1) Bigtable is designed to be a high-scalable, distributed denormalised structure, based on *column families* (groups of columns frequently accessed together). Join operations are expensive (time consuming) and unpredictable at scale, so Bigtable duplicates data to guarantee fast, single-row access.

**Expected output:** samples read from Bigtable should look like:

```
Row key: Australia
severity: case_fatality_ratio -> 0.21
severity: max_total_cases -> 11861161
severity: max_total_deaths -> 25236
health_vax: avg_hosp_patients -> 1708.11
health_vax: avg_icu_patients -> 121.09
health_vax: vax_per_1k -> 2647.56
meta: continent -> Oceania
meta: population -> 26177410
```

#### Creating BigTable Instance

In [20]:
# CODE

INSTANCE_ID = "covid-instance"
TABLE_ID = "covid-summary" 
CLUSTER_ID = "covid-cluster"

# Creating BigTable instance 
instance = client.instance(INSTANCE_ID)
if instance.exists():
    print(f"Instance '{INSTANCE_ID}' already exists.")
else:
    print(f"Creating Bigtable instance '{INSTANCE_ID}'...")
    cluster = instance.cluster(CLUSTER_ID, location_id=ZONE, serve_nodes=1, default_storage_type=enums.StorageType.SSD)
    instance.create(clusters=[cluster])
    print("Instance created successfully!")
    
# get the table reference
table = instance.table(TABLE_ID)

# column family names
COLUMN_FAMILY_META = "meta"
COLUMN_FAMILY_SEVERITY = "severity"
COLUMN_FAMILY_HEALTH_VAX = "health_vax"

# create the table with the specified column families
if not table.exists():
    table.create(column_families={
        COLUMN_FAMILY_META: bigtable.column_family.MaxVersionsGCRule(1),
        COLUMN_FAMILY_SEVERITY: bigtable.column_family.MaxVersionsGCRule(1),
        COLUMN_FAMILY_HEALTH_VAX: bigtable.column_family.MaxVersionsGCRule(1)
    })

    print(f"Table '{TABLE_ID}' created with column families "
          f"'{COLUMN_FAMILY_META}', "
          f"'{COLUMN_FAMILY_SEVERITY}', "
          f"'{COLUMN_FAMILY_HEALTH_VAX}'")
else:
    print(f"Table '{TABLE_ID}' already exists.")


Creating Bigtable instance 'covid-instance'...
Instance created successfully!
Table 'covid-summary' created with column families 'meta', 'severity', 'health_vax'


#### Inserting Spark SQL summary tables

In [21]:
# CODE

# meta info per country - continent & population)
meta_df = (
    covid_df
    .filter(sql_f.col("continent").isNotNull())
    .filter(~sql_f.col("iso_code").startswith("OWID"))
    .select("location", "continent", "population")
    .filter(sql_f.col("population").isNotNull())
    .groupBy("location")
    .agg(
        sql_f.first("continent", ignorenulls=True).alias("continent"),
        sql_f.first("population", ignorenulls=True).alias("population")
    )
)

# Join summary tables in Spark
final_df = (
    meta_df
    .join(severity_df, on="location", how="inner")
    .join(health_vax_df, on="location", how="inner")
    .dropna(subset=[
        "continent", "population",
        "max_total_cases", "max_total_deaths", "case_fatality_ratio",
        "avg_hosp_patients", "avg_icu_patients", "vax_per_1k"
    ])
)

final_df.show(5, truncate=False)


+--------+-------------+----------+---------------+----------------+-------------------+-----------------+----------------+----------+
|location|continent    |population|max_total_cases|max_total_deaths|case_fatality_ratio|avg_hosp_patients|avg_icu_patients|vax_per_1k|
+--------+-------------+----------+---------------+----------------+-------------------+-----------------+----------------+----------+
|Ireland |Europe       |5023108   |1745088        |9744            |0.56               |415.2            |18.03           |2218.18   |
|France  |Europe       |67813000  |38997490       |168091          |0.43               |18085.09         |1986.96         |2276.98   |
|Canada  |North America|38454328  |4819055        |55282           |1.15               |4027.83          |336.3           |2663.79   |
|Japan   |Asia         |123951696 |33803572       |74694           |0.22               |14252.59         |986.88          |3095.85   |
|Israel  |Asia         |9449000   |4841558        |1270

#### Writing to BigTable

In [22]:
# CODE

# Writing to BigTable
ts = datetime.datetime.utcnow()
rows_written = 0
rows = final_df.collect()

for r in rows:
    row_key = r["location"]
    row = table.direct_row(row_key)

    # meta
    row.set_cell("meta", "continent", str(r["continent"]).encode("utf-8"), timestamp=ts)
    row.set_cell("meta", "population", str(r["population"]).encode("utf-8"), timestamp=ts)

    # severity
    row.set_cell("severity", "max_total_cases", str(r["max_total_cases"]).encode("utf-8"), timestamp=ts)
    row.set_cell("severity", "max_total_deaths", str(r["max_total_deaths"]).encode("utf-8"), timestamp=ts)
    row.set_cell("severity", "case_fatality_ratio", str(r["case_fatality_ratio"]).encode("utf-8"), timestamp=ts)

    # health_vax
    row.set_cell("health_vax", "avg_hosp_patients", str(r["avg_hosp_patients"]).encode("utf-8"), timestamp=ts)
    row.set_cell("health_vax", "avg_icu_patients", str(r["avg_icu_patients"]).encode("utf-8"), timestamp=ts)
    row.set_cell("health_vax", "vax_per_1k", str(r["vax_per_1k"]).encode("utf-8"), timestamp=ts)

    row.commit()
    rows_written += 1

print("Rows written to Bigtable:", rows_written)


Rows written to Bigtable: 32


#### Reading from BigTable

In [23]:
# CODE

# Reading a few rows back from BigTable
print("Reading data from Bigtable...")

def read_and_print(country):
    row = table.read_row(country)

    if row is None:
        print(f"\nRow key: {country} (NOT FOUND)")
        return

    print(f"\nRow key: {country}")

    for family in ["severity", "health_vax", "meta"]:
        if family in row.cells:
            for column, cell_values in row.cells[family].items():
                value = cell_values[0].value.decode("utf-8")
                print(f"{family}: {column.decode('utf-8')} -> {value}")

# Sample rows
read_and_print("Australia")
read_and_print("Japan")

Reading data from Bigtable...

Row key: Australia
severity: case_fatality_ratio -> 0.21
severity: max_total_cases -> 11861161
severity: max_total_deaths -> 25236
health_vax: avg_hosp_patients -> 1708.11
health_vax: avg_icu_patients -> 121.09
health_vax: vax_per_1k -> 2647.56
meta: continent -> Oceania
meta: population -> 26177410

Row key: Japan
severity: case_fatality_ratio -> 0.22
severity: max_total_cases -> 33803572
severity: max_total_deaths -> 74694
health_vax: avg_hosp_patients -> 14252.59
health_vax: avg_icu_patients -> 986.88
health_vax: vax_per_1k -> 3095.85
meta: continent -> Asia
meta: population -> 123951696


---

### Question 11: COVID-19 Normalised Impact Score

You were asked to **engineer a composite metric to rank countries** analytically.

Using the country summary Bigtable table created in Question 8, compute a **normalised impact score** for each country, defined as:

```
normalised_burden = ((avg_hosp_patients + avg_icu_patients) / population) * 100000
```

and

```
impact_score = normalised_burden − (vax_per_1k / 1000)
```

and return the top 20 countries with the lowest impact score, sorted in ascending order.

**Requirements & Constraints:**
* For each country row, extract the necessary columns from the relevant column families
* Skip rows with missing required values
* Output must contain exactly 20 rows
* Do not perform joins (Bigtable is denormalised).
* Recall that values are stored as bytes (`utf-8`) in Bigtable, so numeric conversion is necessary before performing calculations

**Expected output:** DataFrame with:
* `country` (or `location`)
* `population`
* `avg_hosp_patients`
* `avg_icu_patients`
* `normalised_burden`
* `vax_per_1k`
* `impact_score`


#### Extracting BigTable Metrics

In [24]:
# CODE

results = []

for row in table.read_rows():

    location = row.row_key.decode("utf-8")
    cells = row.cells
    meta = cells["meta"]
    hv   = cells["health_vax"]
    
    # Skip rows with missing required values
    if (b"population" not in meta or
        b"avg_hosp_patients" not in hv or
        b"avg_icu_patients" not in hv or
        b"vax_per_1k" not in hv):
        continue

    # Decode and numeric conversion of required values
    pop  = float(meta[b"population"][0].value.decode("utf-8"))
    hosp = float(hv[b"avg_hosp_patients"][0].value.decode("utf-8"))
    icu  = float(hv[b"avg_icu_patients"][0].value.decode("utf-8"))
    vax  = float(hv[b"vax_per_1k"][0].value.decode("utf-8"))

    # Calculation of metrics
    normalised_burden = ((hosp + icu) / pop) * 100000
    impact_score = normalised_burden - (vax / 1000)

    # Adding to results list
    results.append((
        location,
        pop,
        hosp,
        icu,
        normalised_burden,
        vax,
        impact_score
    ))

#### Converting to Spark Dataframe

In [26]:
# CODE

# Defining schema for the dataframe
schema = ["location", "population", "avg_hosp_patients", "avg_icu_patients",
          "normalised_burden", "vax_per_1k", "impact_score"]

# Converting list of metrics to Spark DF with above schema
impact_df = spark.createDataFrame(results, schema=schema)

# Adding columns
impact_df = (impact_df
    .withColumn("population", sql_f.col("population").cast("long"))
    .withColumn("avg_hosp_patients", sql_f.round(sql_f.col("avg_hosp_patients"), 2))
    .withColumn("avg_icu_patients", sql_f.round(sql_f.col("avg_icu_patients"), 2))
    .withColumn("normalised_burden", sql_f.round(sql_f.col("normalised_burden"), 4))
    .withColumn("vax_per_1k", sql_f.round(sql_f.col("vax_per_1k"), 2))
    .withColumn("impact_score", sql_f.round(sql_f.col("impact_score"), 4))
)

# Narrowing to top 20 countries with lowest impact score
top20_lowest = impact_df.orderBy(sql_f.col("impact_score").asc()).limit(20)
top20_lowest.show(truncate=False)

+-------------+----------+-----------------+----------------+-----------------+----------+------------+
|location     |population|avg_hosp_patients|avg_icu_patients|normalised_burden|vax_per_1k|impact_score|
+-------------+----------+-----------------+----------------+-----------------+----------+------------+
|Netherlands  |17564020  |607.89           |160.12          |4.3726           |2263.96   |2.1087      |
|Australia    |26177410  |1708.11          |121.09          |6.9877           |2647.56   |4.3401      |
|Denmark      |5882259   |410.33           |27.99           |7.4516           |2542.33   |4.9092      |
|Malaysia     |33938216  |2214.93          |234.62          |7.2177           |2140.87   |5.0768      |
|Luxembourg   |647601    |39.73            |8.22            |7.4043           |2122.52   |5.2817      |
|Bolivia      |12224114  |711.11           |83.22           |6.4981           |1201.77   |5.2963      |
|South Korea  |51815808  |3764.23          |436.24          |8.1

---

### FOLLOW-UP QUESTIONS:

Pick one of these questions and discuss on your report:

* **FQ1.** Explain why a country may have a high vaccination per 1,000 metric but still show high average ICU or hospital occupancy.
* **FQ2.** How does average ICU load correlate (visually or conceptually) with vaccination intensity?
* **FQ3.** Which continents dominate the top CFR rankings? What structural factors (age structure, reporting quality, health capacity) might explain this?
* **FQ4.** What are limitations of the impact score calculated in Question 11?

### FQ4: Limitations of the Impact Score

   The impact score calculation poses several limitations. First, the normalised burden term, defined as the sum of average hospitalised and ICU patients divided by population, does not purely measure disease burden. Hospital patients and ICU patients are also reflective of capacity, as countries with greater hospital infrastructure can also admit more patients, dragging their averages for hospital and icu patients upwards, even when accounting for population. Consequently, the metric may reflect healthcare capacity rather than underlying epidemiological burden. Moreover, both hospitalization and ICU measures themselves depend heavily on the quality of data-reporting by countries, which introduces reporting bias. Countries with greater resources are generally better-equipped to report accurate numbers and more frequently, dragging the normalized burden term upwards once more. By extension, because only countries with certain non-null entries were included, a selection bias is introduced. The data favors countries with greater data reporting capacity which generally tend to be those that are higher-income. As a result, higher-income countries are more represented in this dataset due to the selection criteria.

   Furthermore, the normalized burden term is also structurally problematic in its logic. It assumes an equal weight for hospitalized and ICU patients in quantifying burden, even though an ICU patient indicates greater severity than a hospital patient. ICU admissions generally represent worse infections and thus a greater burden on the healthcare system than a hospital admission. By treating those two categories as mathematically equivalent in weighing, the normalized burden term may overstate burden in countries with higher average hospital patients but relatively few severe cases, while understating burden in countries with greater ICU admissions. The hospitalization measures, themselves, are also affected by the demographic makeup of a country. Countries with more elders, like Italy for example, naturally experience a higher hospitalization and ICU admissions than countries with a relatively younger demographic makeup. While population is accounted for in the normalized burden term, it is missing a key factor of how the population is composed as older populations are more vulnerable to the virus. Thus, cross-country comparisons using raw averages by population may confound disease impact with demographic composition.

   In addition to the limitations of the normalized burden term, the impact score also suffers additional limitations from the vaccination adjustment term. Subtracting vaccinations per 1,000 people scaled by a factor of 1,000, assumes a direct and proportional reduction in the normalized burden term. However, the relationship between vaccination and hospital burden is not necessarily linear as the impact score assumes. It is not realistic to assume that for every additional vaccination, there is a direct linear decrease in the burden on a country. Vaccine administration depends on how many doses were required for a certain vaccine type, whether boosters were readily distributed, and on public health policy concerning vaccinations. Rather than just being a counteractor against burden, it is more reflective of vaccination regimens and government policy than just an effective immunity against hospital burden. Therefore, the subtraction term oversimplifies the complex interaction between vaccination coverage and healthcare burden. Overall, the impact score conflates healthcare capacity, reporting quality, demographic structure, and vaccination policy effects with disease severity. While useful as a simplified indicator, it should not be interpreted as a precise measure of COVID-19’s true impact across countries. 

